# IAN 601 — Module 13: Statistics & Visualization

Module 12 focused on making data trustworthy. Module 13 asks what the cleaned data is telling us.

**Goal:** use statistics and visualizations together to describe distributions, compare groups, investigate relationships, and interpret results carefully.

We will use one cleaned order dataset throughout the notebook.

## 1. Load and inspect the dataset

The dataset contains categorical variables such as `region` and `channel`, and numeric variables such as `amount` and `satisfaction`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data_url = "https://raw.githubusercontent.com/ali2449/IAN-601-Introduction-to-Data-Analytics-Methods-and-Approaches/main/IAN601_Module13_Statistics_Visualization_400.csv"

orders = pd.read_csv(
    data_url,
    parse_dates=["order_date"]
)

orders.head()

In [ ]:
print("Shape:", orders.shape)
print()
print(orders.dtypes)
print()
print("Missing values:")
print(orders.isna().sum())

## 2. Descriptive statistics

Do not stop at one number. Mean, median, standard deviation, minimum, maximum, and quartiles answer different questions.

In [ ]:
orders["amount"].describe()

In [ ]:
print("Mean:", round(orders["amount"].mean(), 2))
print("Median:", round(orders["amount"].median(), 2))
print("Std. dev.:", round(orders["amount"].std(), 2))
print("Minimum:", round(orders["amount"].min(), 2))
print("Maximum:", round(orders["amount"].max(), 2))

## 3. Mean versus median

- **Mean** uses every value and is sensitive to extreme observations.
- **Median** is the middle value and is more resistant to extreme observations.

Comparing both helps us understand the shape of a numeric variable.

In [ ]:
mean_amount = orders["amount"].mean()
median_amount = orders["amount"].median()

print("Mean amount:", round(mean_amount, 2))
print("Median amount:", round(median_amount, 2))
print("Difference:", round(mean_amount - median_amount, 2))

## 4. Visualize a distribution with a histogram

A histogram helps us see where values are concentrated, how spread out they are, and whether the distribution appears symmetric or skewed.

In [ ]:
plt.figure(figsize=(8, 4))

plt.hist(
    orders["amount"],
    bins=20,
    edgecolor="black"
)

plt.xlabel("Order amount")
plt.ylabel("Frequency")
plt.title("Distribution of Order Amounts")
plt.tight_layout()
plt.show()

## 5. Boxplots summarize distributions

A boxplot shows the median, the middle 50% of the data, overall spread, and possible unusual observations.

In [ ]:
plt.figure(figsize=(8, 4))

plt.boxplot(
    orders["amount"],
    vert=False
)

plt.xlabel("Order amount")
plt.title("Boxplot of Order Amounts")
plt.tight_layout()
plt.show()

### Compare distributions across groups

The Module 13 slides compare order amounts across regions. A grouped boxplot makes differences in center and spread easier to see.

In [ ]:
region_order = ["North", "South", "East", "West"]

groups = [
    orders.loc[
        orders["region"] == region,
        "amount"
    ]
    for region in region_order
]

plt.figure(figsize=(8, 4))

plt.boxplot(
    groups,
    tick_labels=region_order
)

plt.ylabel("Order amount")
plt.title("Order Amount by Region")
plt.tight_layout()
plt.show()

## 6. Compare groups with summary statistics

A chart is useful, but we should also calculate the values behind the chart.

In [ ]:
region_summary = (
    orders
    .groupby("region")["amount"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

region_summary

## 7. Bar charts for categorical comparisons

A bar chart is appropriate when we want to compare a summary statistic across categories.

In [ ]:
regional_mean = (
    orders
    .groupby("region")["amount"]
    .mean()
    .sort_values()
)

plt.figure(figsize=(8, 4))

regional_mean.plot(kind="bar")

plt.ylabel("Mean order amount")
plt.xlabel("Region")
plt.title("Average Order Amount by Region")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Scatter plots for numeric relationships

A scatter plot shows how two numeric variables move together.

Here, `amount` is continuous while `satisfaction` is a 1–5 score, so we expect points to appear in horizontal bands.

In [ ]:
plt.figure(figsize=(8, 4))

plt.scatter(
    orders["amount"],
    orders["satisfaction"],
    alpha=0.45
)

plt.xlabel("Order amount")
plt.ylabel("Satisfaction")
plt.title("Order Amount vs Satisfaction")
plt.yticks([1, 2, 3, 4, 5])
plt.tight_layout()
plt.show()

## 9. Correlation

Correlation summarizes the direction and strength of a **linear** relationship.

Values are between -1 and 1:

- near **1** → strong positive linear relationship
- near **-1** → strong negative linear relationship
- near **0** → weak linear relationship

Correlation does **not** prove causation.

In [ ]:
corr = orders[
    ["amount", "satisfaction"]
].corr()

corr

In [ ]:
amount_satisfaction_corr = corr.loc[
    "amount",
    "satisfaction"
]

print(
    "Amount/satisfaction correlation:",
    round(amount_satisfaction_corr, 3)
)

## 10. Correlation can be misleading

A correlation coefficient is only one summary.

Before interpreting correlation, also ask:

- Does the scatter plot support the same story?
- Are there unusual values?
- Is the relationship actually linear?
- Could another variable explain the pattern?
- Does the sample represent the population we care about?

A relationship can be real without being causal.

## 11. A simple confidence interval idea

A confidence interval gives a range of plausible values for a population parameter.

For this introductory example, we use a large-sample approximation for the population mean order amount.

In [ ]:
import math

x = orders["amount"]

sample_mean = x.mean()
standard_error = x.std(ddof=1) / math.sqrt(len(x))
margin = 1.96 * standard_error

lower = sample_mean - margin
upper = sample_mean + margin

print("Sample mean:", round(sample_mean, 2))
print(
    "Approximate 95% confidence interval:",
    round(lower, 2),
    "to",
    round(upper, 2)
)

The confidence interval is about the **population mean**, not the range where 95% of individual order amounts must fall.

## 12. Choose the right visual

| Question | Useful visual |
|---|---|
| What does one numeric distribution look like? | Histogram |
| What are the median, spread, and possible unusual values? | Boxplot |
| How do categories compare? | Bar chart |
| How do two numeric variables relate? | Scatter plot |
| How does a value change over time? | Line chart |

Choose the chart based on the analytical question, not because a chart type looks interesting.

## 13. Make charts easier to read

A useful chart should have:

- a clear title,
- meaningful axis labels,
- readable scales,
- appropriate category labels,
- minimal unnecessary decoration.

The reader should understand the main question without needing to inspect the code.

## 14. Line charts for time trends

Because several orders may occur on the same date, we first calculate daily revenue and then plot it in chronological order.

In [ ]:
daily_revenue = (
    orders
    .groupby("order_date", as_index=False)["amount"]
    .sum()
    .sort_values("order_date")
)

daily_revenue.head()

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(
    daily_revenue["order_date"],
    daily_revenue["amount"]
)

plt.xlabel("Date")
plt.ylabel("Daily revenue")
plt.title("Daily Revenue Trend")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 15. Mini statistics report

A useful analytical report often combines a numerical summary and a visual comparison.

In [ ]:
report = (
    orders
    .groupby("region")["amount"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

report

In [ ]:
orders.boxplot(
    column="amount",
    by="region",
    grid=False,
    figsize=(8, 4)
)

plt.suptitle("")
plt.title("Order Amount by Region")
plt.xlabel("Region")
plt.ylabel("Order amount")
plt.tight_layout()
plt.show()

## 16. Interpret carefully

A good interpretation connects the numerical evidence with the visualization.

For example:

- Which region has the highest average order amount?
- Is the median telling the same story as the mean?
- Are the spreads similar across regions?
- Does the chart reveal anything the summary table hides?

Use language such as **associated with**, **higher on average**, or **the data suggests** when the analysis does not establish causation.

In [ ]:
highest_mean_region = report["mean"].idxmax()
highest_mean_value = report.loc[
    highest_mean_region,
    "mean"
]

print(
    "Highest mean order amount:",
    highest_mean_region,
    "->",
    highest_mean_value
)

print(
    "Amount/satisfaction correlation:",
    round(amount_satisfaction_corr, 3)
)

## Practice challenge

Use the `orders` DataFrame.

1. Display descriptive statistics for `amount`.
2. Compare the mean and median.
3. Create a histogram of `amount`.
4. Summarize `amount` by region.
5. Create a grouped boxplot by region.
6. Create a bar chart comparing average amount by channel.
7. Plot amount versus satisfaction.
8. Calculate their correlation.
9. Create a line chart of **daily total revenue**.
10. Write two or three sentences interpreting one result without claiming causation.

In [ ]:
# 1. Descriptive statistics


# 2. Mean and median


# 3. Histogram


# 4. Summary by region


# 5. Grouped boxplot


# 6. Bar chart by channel


# 7. Scatter plot


# 8. Correlation


# 9. Daily revenue trend


# 10. Interpretation as comments


## Module 13 complete

You have practiced using numerical summaries and visual evidence together.

Module 14 builds on this foundation by using features to predict a numeric target, starting with **house-price regression** and a train/test workflow.